In [1]:
import sys
from pathlib import Path 

project_root = Path().resolve().parent  
sys.path.append(str(project_root))

In [2]:
import pandas as pd
import numpy as np

from src.model_openaiapi import sent_analysis_openai

In [4]:
df_cars = pd.read_csv('../data/r_cars_reddit_comments.csv')
df_cars = df_cars[['body']]

df_cars = df_cars.rename(columns={'body':'text'})

print(f"df shape: {df_cars.shape}")
df_cars.head()

df shape: (197, 1)


,text
0,"Some key quotes I think:\n\n&gt;""The American ..."
1,Was the problem MSRP or greedy dealerships jac...
2,Both.
3,I'm sure it has nothing to do with the America...
4,If I’m spending $40k+ on a car it fuckin bette...


In [5]:
def clean_oai_df(df, prompt, max_retries = 30, attempt = 0):
    """
    Reruns the OpenAI API calls for records with missing preditions (NaN) until all missing values are filled or the maximum retry limit is reached.

    Input: 
        df: validation dataset with sentiment predictions made by the GPT 5.1 model 
        prompt: Prompt for inference

    Output: complete dataset with missing values handled
    """

    # Iterate until prediction is made for all NaN records. Max attempts used to avoid inf loop
    while df['sentiment_pred_openai'].isna().sum() > 0 and attempt < max_retries:
        
        attempt += 1

        # Identiy missing (NaN) records, get total count and create boolean mask for NaN records
        missing_count = df['sentiment_pred_openai'].isna().sum()
        mask_df = df['sentiment_pred_openai'].isna()
        
        # Use boolean mask to get df subset with NaN records
        df_nan = df.loc[mask_df].copy()
        
        # Rerun OpenAI API call to generate sentiment prediction for missing records
        df_nan = df_nan.drop(columns=['input_text', 'sentiment_pred_openai'])
        df_new_pred = sent_analysis_openai(df_nan, prompt, batch_size=missing_count)

        # Update df
        cols_to_update = ['sentiment_pred_openai', 'input_text']
        df.loc[mask_df, cols_to_update] = df_new_pred[cols_to_update].values

        # Replace 'nan' with NaN
        df = df.replace('nan', np.nan)

    return df


In [6]:
fewshot_prompt = (
    "You are a sentiment analyzer for automotive Reddit data.\n"
    "Your task is to classify text from the r/cars subreddit as positive, negative, or neutral. "
    "You must consider the specific context, enthusiast slang, and common reddit opinions related to car brands, reliability, and driving dynamics.\n"
    "\n"
    "--- Examples (Few-Shot Demonstrations) ---\n"
    "('The handling on the new Miata is perfection; it’s the best driver’s car for the money.', positive)|||"
    "('I am so tired of every brand replacing physical buttons with laggy touchscreens. It is dangerous and cheap.', negative)|||"
    "('The base model Camry is a reliable commuter car that gets the job done without any fuss.', neutral)|||"
    "('If Mazda actually brings back the rotary engine as a range extender, it could be a game changer for their EVs.', positive)|||"
    "('The build quality on these early production units is a total disaster, panels gaps everywhere.', negative)\n"
    "--- End of Examples ---\n"
    "\n"
    "Classify each new input text below as positive, negative, or neutral.\n"
    "Each input text is separated by '|||'.\n"
    "Return exactly one tuple for EACH input text, in the SAME ORDER, following the examples above.\n"
    "Each tuple must have this exact format: (original_text, sentiment_label)\n"
    "sentiment_label must be one of: positive, negative, neutral.\n"
    "Tuples must be separated by '|||'.\n"
    "Do NOT add or remove any text, punctuation, spaces, or line breaks.\n"
)

In [7]:
# Make sentiment predictions using the OpenAI API 
df_oai_fs = df_cars.copy()
df_oai_fs_pred = sent_analysis_openai(df_oai_fs, fewshot_prompt, 10)
df_final_pred_fs_oai = clean_oai_df(df_oai_fs_pred, fewshot_prompt)
print(df_final_pred_fs_oai['sentiment_pred_openai'].value_counts(dropna=False))

sentiment_pred_openai
neutral     83
negative    80
positive    34
Name: count, dtype: int64


In [9]:
df_final_pred_fs_oai = df_final_pred_fs_oai.rename(columns={'text':'comment','sentiment_pred_openai':'comment sentiment'})
df_final_pred_fs_oai = df_final_pred_fs_oai[['comment','comment sentiment']]

df_final_pred_fs_oai.head()

,comment,comment sentiment
0,"Some key quotes I think:\n\n&gt;""The American ...",negative
1,Was the problem MSRP or greedy dealerships jac...,neutral
2,Both.,neutral
3,I'm sure it has nothing to do with the America...,negative
4,If I’m spending $40k+ on a car it fuckin bette...,neutral


In [10]:
output_path = "../data/cars_subreddit_sentanalysis.csv"
df_final_pred_fs_oai.to_csv(output_path, index=False)

print("File exported")

File exported
